# Biohub leakage-safe CV — evaluate saved folds

Loads the checkpoints created by the two temporal-CV training notebooks, predicts only each held-out embryo, and reports the official tracking metric. No model training occurs here.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

COMP = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAIN = COMP / 'train'
INPUT = Path('/kaggle/input')
WEIGHTS_INPUT_CANDIDATES = [INPUT / 'biohub-cv-temporal-fold-weights-v1', INPUT / 'datasets/versavice/biohub-cv-temporal-fold-weights-v1']
WEIGHTS_INPUT = next((path for path in WEIGHTS_INPUT_CANDIDATES if path.exists()), WEIGHTS_INPUT_CANDIDATES[0])
SUPPORT_CANDIDATES = [INPUT / 'biohub-tracking-support-pack-50ep-v1', INPUT / 'datasets/pilkwang/biohub-tracking-support-pack-50ep-v1']
SUPPORT = next((path for path in SUPPORT_CANDIDATES if path.exists()), SUPPORT_CANDIDATES[0])
WORK = Path('/kaggle/working')
REPO = WORK / 'tracking_repo'
assert TRAIN.exists(), TRAIN
assert SUPPORT.exists(), SUPPORT
assert WEIGHTS_INPUT.exists(), WEIGHTS_INPUT
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SUPPORT / 'repo', REPO)
packages = ['tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff-spec', 'ilpy', 'polars-runtime-32', 'polars', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy', 'donfig', 'numcodecs']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--find-links', str(SUPPORT / 'wheels'), '--force-reinstall', *packages], check=True)
import polars as pl
assert hasattr(pl, 'Float16'), pl.__version__


In [ ]:
def embryo_id(name: str) -> str:
    return name.removesuffix('.zarr').split('_', 1)[0]

samples = sorted(path.name.removesuffix('.zarr') for path in TRAIN.glob('*.zarr'))
assert sorted({embryo_id(name) for name in samples}) == ['44b6', '6bba']
folds = [
    {'name': 'fold0_44b6_to_6bba', 'train_embryo': '44b6', 'held_out_embryo': '6bba'},
    {'name': 'fold1_6bba_to_44b6', 'train_embryo': '6bba', 'held_out_embryo': '44b6'},
]
for fold in folds:
    fold['train'] = [name for name in samples if embryo_id(name) == fold['train_embryo']]
    fold['test'] = [name for name in samples if embryo_id(name) == fold['held_out_embryo']]
splits_path = REPO / 'dataset_splits.json'
splits_path.write_text(json.dumps(folds, indent=2))
print([(fold['name'], len(fold['train']), len(fold['test'])) for fold in folds])


In [ ]:
# The private Dataset stores each fold as a zip to preserve its directory name.
EXTRACTED = WORK / 'cv_weights'
EXTRACTED.mkdir(exist_ok=True)
for archive in sorted(WEIGHTS_INPUT.glob('fold*.zip')):
    shutil.unpack_archive(archive, EXTRACTED)
# Kaggle may expose Dataset zip files either intact or already expanded.
all_weights = sorted(set(EXTRACTED.glob('**/edge_predictor_best.pth')) | set(WEIGHTS_INPUT.glob('**/edge_predictor_best.pth')))
all_configs = sorted(set(EXTRACTED.glob('**/config.json')) | set(WEIGHTS_INPUT.glob('**/config.json')))
print('Attached checkpoints:', [str(path) for path in all_weights])

def source_file(slug: str, files: list[Path]) -> Path:
    matches = [path for path in files if slug in str(path)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {slug} file, found: {matches}')
    return matches[0]

sources = {
    0: 'fold0',
    1: 'fold1',
}
for fold_index, slug in sources.items():
    weight = source_file(slug, all_weights)
    config = source_file(slug, all_configs)
    target = REPO / 'weights/unet_transformer' / f'split_{fold_index}'
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weight, target / 'edge_predictor_best.pth')
    shutil.copy2(config, target / 'config.json')
    print(f'fold {fold_index}: {weight}')


In [ ]:
# First run is deliberately bounded: it verifies that both fold paths work
# before spending hours on all held-out clips. Set SMOKE_ONLY=False for full CV.
SMOKE_ONLY = False
FOLD_INDICES = (0,)
PREDICTED_COUNTS = {}
for fold_index in FOLD_INDICES:
    cmd = [
        sys.executable, 'scripts/predict_unet_transformer.py',
        '--data-dir', str(TRAIN), '--splits', 'dataset_splits.json', '--split', str(fold_index),
        '--weights', f'weights/unet_transformer/split_{fold_index}/edge_predictor_best.pth',
        '--unet-batch-size', '4', '--det-threshold', '0.99', '--use-ilp',
    ]
    if SMOKE_ONLY:
        cmd += ['--slice', ':2']
    log_path = WORK / f'predict_fold{fold_index}.log'
    with log_path.open('w') as log:
        subprocess.run(cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'BIOHUB_DATA_DIR': str(TRAIN), 'PYTORCH_ALLOC_CONF': 'expandable_segments:True'}, stdout=log, stderr=subprocess.STDOUT, check=True)
    output_dir = REPO / 'predictions/unknown/unet_transformer' / f'split_{fold_index}'
    PREDICTED_COUNTS[fold_index] = len(list(output_dir.glob('*.geff')))
    expected = min(2, len(folds[fold_index]['test'])) if SMOKE_ONLY else len(folds[fold_index]['test'])
    print(f'fold {fold_index}: predicted {PREDICTED_COUNTS[fold_index]} / expected {expected}')
    assert PREDICTED_COUNTS[fold_index] == expected, (fold_index, PREDICTED_COUNTS[fold_index], expected)

eval_cmd = [sys.executable, 'scripts/evaluate.py', '--method', 'unet_transformer', '--split', 'all']
eval_log = WORK / 'evaluate.log'
with eval_log.open('w') as log:
    subprocess.run(eval_cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'BIOHUB_DATA_DIR': str(TRAIN)}, stdout=log, stderr=subprocess.STDOUT, check=True)
print(eval_log.read_text()[-12000:])


In [ ]:
# Save all machine-readable evidence as Notebook output.
results_dir = REPO / 'results'
for path in sorted(results_dir.rglob('*')) if results_dir.exists() else []:
    if path.is_file():
        print(path.relative_to(REPO))
(WORK / 'cv_evaluation_manifest.json').write_text(json.dumps({'smoke_only': SMOKE_ONLY, 'predicted_counts': PREDICTED_COUNTS, 'folds': folds}, indent=2))
